<a href="https://colab.research.google.com/github/pavithrasmv/sqlite/blob/main/sqldatacleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import pandas as pd
import sqlite3
df = pd.read_excel("/content/orders_raw.xlsx")
conn = sqlite3.connect("order_raw.db")

df.to_sql("order_raw", conn, if_exists="replace", index=False)

print(pd.read_sql_query( "SELECT COUNT(*) AS tot_no_of_rows FROM order_raw",conn))
print(pd.read_sql_query("SELECT * FROM order_raw LIMIT 10",conn))
conn.commit()

print("-------------------------------------------------------------------")
print("---------------------- Check duplicate OrderID values-------------- ")
print(pd.read_sql_query("""SELECT
OrderID,
COUNT(*) AS Duplicate_Count
FROM order_raw
GROUP BY OrderID
HAVING COUNT(*) > 1
ORDER BY Duplicate_Count DESC""",conn))


print("-------------------------------------------------------------------")
print("---------------------Check missing values -------------------------")
print(pd.read_sql_query(""" SELECT
SUM(CASE WHEN CustomerName IS NULL OR TRIM(CustomerName)='' THEN 1 ELSE 0 END)AS Missing_CustomerName,
SUM(CASE WHEN Email IS NULL OR TRIM(Email)='' THEN 1 ELSE 0 END )AS Missing_Email,
SUM(CASE WHEN Phone IS NULL OR TRIM(Phone)='' THEN 1 ELSE 0 END)AS Missing_Phone,
SUM(CASE WHEN City IS NULL OR TRIM(City)='' THEN 1 ELSE 0 END)AS Missing_City,
SUM(CASE WHEN OrderDate IS NULL OR TRIM(OrderDate)='' THEN 1 ELSE 0 END)AS Missing_OrderDate,
SUM(CASE WHEN Amount IS NULL OR TRIM(Amount)='' THEN 1 ELSE 0 END)AS Missing_Amount,
SUM(CASE WHEN Status IS NULL OR TRIM(Status)='' THEN 1 ELSE 0 END) AS Missing_Status
FROM order_raw""",conn
))
conn.commit()


print("------------------------------------------------------------------")
print("---------------------- Create a staging table----------------------")
conn.execute("DROP TABLE IF EXISTS orders_staging")
conn.execute("""CREATE TABLE orders_staging AS SELECT * FROM order_raw""")
conn.commit()
print(pd.read_sql_query("SELECT COUNT(*) AS Staging_Rows FROM orders_staging ",conn))


print("------------------------------------------------------------------")
print("----------------------Remove exact duplicate rows ---------------")
conn.execute("""DELETE FROM orders_staging WHERE rowid NOT IN(SELECT MIN(rowid)
FROM orders_staging
GROUP BY
OrderID,
CustomerName,
Email,
Phone,
City,
OrderDate,
Amount,
Status)""")
conn.commit()
print(pd.read_sql_query("""SELECT COUNT(*) AS Rows_After_Duplicate_Removal FROM orders_staging """,conn))




print("----------------------------------------------------------------")
print("----------------------Remove unwanted spaces -------------------")
conn.execute("""UPDATE orders_staging SET CustomerName=TRIM(CustomerName),Email=TRIM(Email),Phone=TRIM(Phone),
City=TRIM(City),OrderDate=TRIM(OrderDate),Status=TRIM(Status),Amount=Trim(Amount)""")
conn.commit()
print(pd.read_sql_query("SELECT * FROM orders_staging LIMIT 10",conn))



print("----------------------------------------------------------------")
print("-----------------Standardize Customer Names ----------------")
conn.execute("""UPDATE orders_staging SET CustomerName=UPPER(SUBSTR(CustomerName,1,1))|| LOWER(SUBSTR(CustomerName, 2, INSTR(CustomerName, ' ') - 2))||' ' || UPPER(SUBSTR(CustomerName,INSTR(CustomerName,' ')+1,1))|| LOWER(SUBSTR(CustomerName,INSTR(CustomerName,' ')+2)) WHERE CustomerNAME IS NOT NULL and INSTR(CustomerName,'')>0""")
conn.commit()
print(pd.read_sql_query("SELECT * FROM orders_staging LIMIT 10",conn))



print("-----------------------------------------------------------------")
print("----------------Standardize City names --------------------------")
conn.execute("""UPDATE orders_staging SET City= CASE UPPER(TRIM(City))
WHEN 'CHENNAI' THEN 'Chennai'
WHEN 'BENGALURU' THEN 'Bengaluru'
WHEN 'COIMBATORE' THEN 'Coimbatore'
WHEN 'MUMBAI' THEN 'Mumbai'
WHEN 'DELHI' THEN 'Delhi'
WHEN 'HYDERABAD' THEN 'Hyderabad'
WHEN 'PUNE' THEN 'Pune'
WHEN 'KOLKATA' THEN 'Kolkata'
ELSE City
END WHERE City IS NOT NULL""")
conn.commit()
print(pd.read_sql_query("SELECT * FROM orders_staging LIMIT 10",conn))


print("-----------------------------------------------------------------")
print("------------------Convert Email to lowercase --------------------")
conn.execute("""UPDATE orders_staging SET Email=LOWER(TRIM(Email)) WHERE Email IS NOT NULL""")
conn.commit()
print(pd.read_sql_query("SELECT * FROM orders_staging LIMIT 10",conn))



print("-----------------------------------------------------------------")
print("---------------------Standardize Status -------------------------")
conn.execute("""UPDATE orders_staging SET Status= CASE UPPER(TRIM(Status))
WHEN 'COMPLETED' THEN 'Completed'
WHEN 'CANCELLED' THEN 'Cancelled'
WHEN 'CANCELED' THEN 'Canceled'
WHEN 'PENDING' THEN 'Pending'
WHEN 'REFUNDED' THEN 'Refunded'
WHEN 'N/A' THEN 'NULL'
WHEN '' THEN 'NULL'
ELSE Status
END""")
conn.commit()
print(pd.read_sql_query("SELECT * FROM orders_staging LIMIT 10",conn))

print("------------------------------------------------------------------")
print(pd.read_sql_query("SELECT Status,COUNT(*)AS tot from orders_staging GROUP BY Status ORDER BY tot DESC",conn))


print("-------------------------------------------------------------------")
print("--------------------Handle missing values -------------------------")
conn.execute("""UPDATE orders_staging SET CustomerName='Unknown Customer' WHERE CustomerName IS NULL OR TRIM(CustomerName)='' """)
conn.execute("""UPDATE orders_staging SET  City='Unknown' WHERE City  IS NULL OR TRIM(City)='' """)
conn.execute("""UPDATE orders_staging SET Status='Unknown' WHERE Status IS NULL OR TRIM(Status)='' """)
conn.commit()
print(pd.read_sql_query("SELECT * FROM orders_staging LIMIT 20",conn))



print("-------------------------------------------------------------------")
print("----------------------Standardize OrderDate------------------------")
conn.execute("""UPDATE orders_staging SET OrderDate=CASE
WHEN OrderDate LIKE '____-__-__' THEN OrderDate
WHEN OrderDate LIKE '__-__-____' THEN SUBSTR(OrderDate,7,4)||'-'||SUBSTR(OrderDate,1,2)||'-'||SUBSTR(OrderDate,4,2)
WHEN OrderDate LIKE '__-__-____' THEN SUBSTR(OrderDate,7,4)||'-'||SUBSTR(OrderDate,4,2)||'-'||SUBSTR(OrderDate,1,2)
ELSE NULL
END WHERE OrderDate IS NOT NULL""")
conn.commit()
print(pd.read_sql_query("SELECT * FROM orders_staging LIMIT 20",conn))





   tot_no_of_rows
0           10000
   OrderID        CustomerName                            Email  \
0     3105          Jaya Verma           jaya.verma801@mail.com   
1     6354        Farhan Yadav     farhan.yadav65@company.co.in   
2     8690       Chitra Reddy      chitra.reddy695company.co.in   
3     5858          Sneha Iyer      sneha.iyer945@company.co.in   
4     6012       kabir kumar         kabir.kumar305@outlook.com   
5     2653        Kabir Joshi            kabir.joshi17@mail.com   
6     5130  Deepa Chatterjee    deepa.chatterjee695@outlook.com   
7      416           Esha Iyer           esha.iyer391@gmail.com   
8     3539         Arun Yadav            arun.yadav764@mail.com   
9     1704       qadir singh            qadir.singh378@mail.com   

            Phone           City   OrderDate    Amount     Status  
0    733-307-7375          delhi  2024-01-05   3091.09    Pending  
1     23114 80745      Bengaluru  04-14-2023   1662.89   Refunded  
2    379-490-1038     